# Lab 7: Train and Diagnose a Network

**CS 463 · Due Monday, Nov 23, 8:30 AM in Canvas**

In our previous lesson we built every piece of training by hand: the log-loss, backpropagation, the gradient descent loop, and minibatches ([Training a Network](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb)). In this lab, scikit-learn's `MLPClassifier` does the backpropagation and the loop. Your job is to choose its settings, train networks with it, and use the loss curve and the accuracy to diagnose training problems. You'll diagnose three kinds of problem:

1. In the first, training stops while the loss is still on a plateau, before the network has learned anything. 
2. In the second, training gets stuck at a high loss, and more epochs don't help. 
3. In the third, a kind of "bottleneck" dooms the network, so even the training accuracy stays low. You'll analyze this important scenario.

For most of this assignment, you'll work with Wednesday's XOR data: 100 points in four clusters, small enough that each network trains in under a second. Then you'll move to handwritten digits, in two versions:

- scikit-learn's 8 × 8 digits are the easy version: each image was shrunk from 32 × 32 pixels by counting the dark pixels in each 4 × 4 block, which smooths out small differences in handwriting. 
- MNIST's 28 × 28 images are the raw pixels, and they're much harder. 


### Overview

1. **Wednesday's network, in scikit-learn:** the same 2-2-1 sigmoid network, and where its 9 parameters live.
2. **"It finished, but it's wrong":** a run that stops on a plateau.
3. **Stuck runs and the loss surface:** why some starts fail, and why more hidden units help.
4. **Minibatches:** epochs vs. gradient steps.
5. **Digits:** a 10-class problem, on 8 × 8 (easy) images.
6. **Harder digits (MNIST):** how "wide" the hidden layer must be, and when a network beats logistic regression.

### How this lab works

- **Work through the parts in order.** Later parts reuse earlier ones.
- **Predict, then run.** Several items ask for a prediction first. Write it down before you run the cell; a wrong prediction you then explain is worth more than a right guess.
- **Hints are folded up** under the items. Open one when you're stuck.
- **What you turn in:** every item labeled by part and letter, like **2(b)**, marked **Code** or **Written**:
  - **Code:** fill in the cell right below, which starts with `# 2(b)`. Where part of the cell is written for you, fill in the part marked `# YOUR CODE HERE` and leave the rest.
  - **Written:** replace `YOUR ANSWER HERE` in the **2(b) Answer:** cell. A few sentences at most.
- **Some cells take a while** (10–30 seconds). Each one says so.
- **Seeds and versions.** Numbers quoted "in our results" come from our execution of this notebook with scikit-learn 1.9.1. A different version (on Colab, say) can give slightly different numbers. Report yours; the patterns are what matter.
- **Graded on completion and good faith,** like every lab. The **Optional** sections are not graded.

**Checklist**

| Part | Items |
|---|---|
| 1 | 1(a) Code |
| 2 | 2(a) Code + Written, 2(b) Code + Written, 2(c) Code, 2(d) Written |
| 3 | 3(a) Code, 3(b) Code + Written |
| 4 | 4(a) Written, 4(b) Written, 4(c) Written |
| 5 | 5(a) Code + Written, 5(b) Code + Written, 5(c) Written, 5(d) Written |
| 6 | 6(a) Code, 6(b) Written, 6(c) Written |

## Setup

This lab uses the `CS_463` environment; there's nothing new to install. It needs two files next to the notebook: `ch_6_and_10_lab_7_helpers.py` (plotting helpers; you don't need to read it) and `mnist.npz` (the data for Part 6). If you pulled the whole `Labs/Ch_6_and_10` folder, you have both. **On Colab,** upload both files first (Files panel, then Upload).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ch_6_and_10_lab_7_helpers import (xor_data, plot_loss_curves, show_digits, plot_confusion,
                                       plot_accuracy_vs_width, plot_h_space, wednesday_start, fit_from_start)

X_xor, y_xor = xor_data()
print(X_xor.shape, y_xor.shape)

## Part 1: Wednesday's network, in scikit-learn

Wednesday's network had 2 inputs, 2 sigmoid hidden units, and 1 sigmoid output, trained by full-batch gradient descent on the log-loss with learning rate $\eta = 1$ ([Training a Network, §6](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-6)). Here it is as an `MLPClassifier`.

**Code worth reading.** This lab builds several versions of a network that differ in one or two settings, so instead of repeating every setting each time, we define a function that creates the default model (the XOR nueral network from our previous lesson). `xor_net(seed)` returns an *unfitted* `MLPClassifier` with Wednesday's settings. Its optional keyword arguments override the default settings, so `xor_net(0, batch_size=10)` is the same network with batches of 10.

In [ ]:
def xor_net(seed, **changes):
    "Wednesday's 2-2-1 sigmoid network. Keyword arguments replace individual settings."
    settings = dict(
        hidden_layer_sizes=(2,),     # one hidden layer with 2 units
        activation='logistic',       # the sigmoid, in the hidden layer (the output is always sigmoid here)
        solver='sgd',                # plain gradient descent...
        momentum=0.0,                # ...with no momentum (sgd's default adds some; Wednesday's didn't)
        learning_rate='constant',    # the same step size every step
        learning_rate_init=1.0,      # eta
        batch_size=100,              # all 100 points per step: full batch
        alpha=0.0,                   # no penalty term (alpha is a penalty on the weights; Monday's topic)
        max_iter=3000,               # at most 3,000 epochs
        tol=1e-4,                    # an epoch counts as an improvement only if it beats the best loss so far by tol
        n_iter_no_change=10,         # stop once more than 10 epochs in a row fail to improve
        random_state=seed,           # the starting weights (and the shuffling)
    )
    settings.update(changes)
    return MLPClassifier(**settings)


xor_default = xor_net(seed=0).fit(X_xor, y_xor)
xor_default

<details>
<summary>The MLPClassifier output (above)</summary>

Displaying a model in a notebook shows scikit-learn's summary of it. Click **MLPClassifier** to open it: **Parameters** lists every setting, with the ones `xor_net` changed from the defaults highlighted, and **Fitted attributes** lists what `.fit` computed, such as `coefs_` and `n_iter_`. The **?** links to the documentation, and **Fitted** confirms the model has been trained. (If you see plain text instead, that's the same summary without the interactive parts.)

</details>

You wrote this network's forward pass yourself in the [coding Learning Check](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_02_coding_learning_check.ipynb), so this lab lets scikit-learn compute it: `xor_default.predict_proba(X)[:, 1]` is the network's $p$ for every row of `X`, and `xor_default.predict(X)` thresholds it at 0.5. Calling `.fit` runs Wednesday's whole training loop: for each epoch, a forward pass, backpropagation, and the update rule.

After fitting, scikit-learn stores the weights in `xor_default.coefs_` and the biases in `xor_default.intercepts_`. Each of these is a Python list that holds one NumPy array per layer: element 0 is for the hidden layer, and element 1 is for the output layer.

**1(a) · Code.** Print the shape of every array in `xor_default.coefs_` and `xor_default.intercepts_`, and compute `n_params`, the total number of weights and biases. It should match the 9 parameters of Wednesday's network.

**Points back to:** [Neurons to Networks, Notation](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_02_neurons_to_networks.ipynb#notation) (9 parameters); [Training a Network, §1](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-1).

<details>
<summary>Hint 1: what the lists hold</summary>

`coefs_` is a list of NumPy arrays: `coefs_[0]` holds the weights into the hidden layer, and `coefs_[1]` the weights into the output layer. `intercepts_` holds the biases the same way. Each array's `.shape` is a tuple, and its `.size` is its number of entries. A list comprehension prints all the shapes at once.

</details>

<details>
<summary>Hint 2: a made-up example</summary>

A network with 4 inputs and 3 hidden units, on made-up data:

```python
X_toy = np.random.default_rng(0).normal(size=(20, 4))      # 20 points, 4 features
y_toy = (X_toy[:, 0] > 0).astype(int)
toy_net = MLPClassifier(hidden_layer_sizes=(3,), max_iter=50, random_state=0).fit(X_toy, y_toy)
print([W.shape for W in toy_net.coefs_])
print([b.shape for b in toy_net.intercepts_])
```
```
[(4, 3), (3, 1)]
[(3,), (1,)]
```

</details>

In [ ]:
# 1(a)
### YOUR CODE HERE


<details>
<summary>Optional: how scikit-learn's arrays line up with Wednesday's notation</summary>

You don't need this for the rest of the lab; it matters if you ever read a network's weights yourself. Run 1(a) first.

The shapes are `coefs_[0]`: (2, 2), `coefs_[1]`: (2, 1), `intercepts_[0]`: (2,), and `intercepts_[1]`: (1,). That's $4 + 2 + 2 + 1 = 9$ parameters, as on Wednesday.

- **The biases match directly:** `intercepts_[0]` is $\mathbf{b}^{(1)}$ (one bias per hidden unit), and `intercepts_[1]` is $b^{(2)}$ (the output unit's bias).
- **Our convention for weights:** in $W^{(1)}$, each **row** holds one hidden unit's weights, and each **column** corresponds to one input feature. Entry $w^{(1)}_{kj}$ is the weight into hidden unit $k$ from input $j$. Likewise, $W^{(2)}$ has one row for the single output unit and one column per hidden unit, so it's $1 \times 2$.
- **scikit-learn's convention is the transpose:** in `coefs_[0]`, each **row** corresponds to one input feature, and each **column** holds one hidden unit's weights. Entry `coefs_[0][j, k]` is the weight from input $j$ into hidden unit $k$, which is $w^{(1)}_{kj}$. So `coefs_[0].T` is $W^{(1)}$. In the same way, `coefs_[1]` has one row per hidden unit and one column for the output unit, so it's $2 \times 1$, and `coefs_[1].T` is $W^{(2)}$.
- **Both weight arrays are stored transposed.** For `coefs_[0]` the shape alone doesn't show it, because $W^{(1)}$ is square ($2 \times 2$); `coefs_[1]`'s shape, $2 \times 1$ instead of $1 \times 2$, does.

</details>

## Part 2: "It finished, but it's wrong"

`xor_default` was allowed up to 3,000 epochs (see settings where we defined it). Wednesday's network solved XOR, but its loss sat nearly flat for long stretches before dropping ([Training a Network, §6](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-6)).

Throughout this lab, a **run** is one training of one model: a single call to `.fit`, from its starting weights through its last epoch.

**2(a) · Code + Written. Predict, then run.** *Predict first:* how many of its 3,000 allowed epochs did `xor_default` complete? Write your guess in the answer cell. Then print `xor_default.n_iter_` (the number of epochs it ran), its final training loss `xor_default.loss_curve_[-1]`, and its accuracy on the XOR data, `xor_default.score(X_xor, y_xor)`.

**Points back to:** [Training a Network, §6](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-6).

In [ ]:
# 2(a)
### YOUR CODE HERE


**2(a) Answer:**

_YOUR ANSWER HERE_


**A helper for loss curves.** From here on you'll plot training-loss curves with `plot_loss_curves`, from the helper file. It takes a **list** of curves, so you can compare several on one set of axes:

| Argument | What to pass |
|---|---|
| `curves` | a list of 1-D sequences of losses: usually each fitted model's `loss_curve_` (one loss per epoch). For one curve, a one-item list: `[xor_default.loss_curve_]`. The curves might have different lengths. |
| `labels` | optional: a list of strings, one per curve, for the legend. Leave it out for many curves. |
| `x` | `'epoch'` (the default) or `'step'` (Part 4). |
| `steps_per_epoch` | with `x='step'`: a list with one number per curve, the gradient steps per epoch. |
| `baseline` | optional: a number drawn as a dashed horizontal line. |
| `title` | the plot's title. |

It returns the matplotlib `Axes`. A dot marks where each curve ends, so a run that stopped early stays visible. For example, two made-up curves:

```python
model_one_losses = [0.9, 0.6, 0.4, 0.3]   # you'll do something like some_model.loss_curve_
model_two_losses = [0.9, 0.8, 0.75]       # i.e. some_other_model.loss_curve_ (lengths may differ)

plot_loss_curves(curves=[model_one_losses, model_two_losses], labels=['fast', 'slow'], title='Two made-up curves')
```

**2(b) · Code + Written. Diagnose.** Plot `xor_default`'s loss curve with a baseline at $\ln 2$ (`np.log(2)`). Then answer:

1. What value is the loss sitting at, and what does that value mean for this network's predictions?
2. Is this run stuck, on a plateau, or neither? How do you know?
3. What ended training after so few epochs? Name the settings responsible, and explain why they ended this run.

**Points back to:** [Training a Network, §1](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-1) (the loss of a network that knows nothing) and [§6](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-6) ("Reading the run").

<details>
<summary>Hint for question 1</summary>

What is the log-loss of predicting $p = 0.5$ for every point?

</details>

<details>
<summary>Hint for question 3: where to look</summary>

Look back at how `xor_net` defines the network in Part 1. Do any of its settings seem relevant? Compare what they do with what the loss curve does on a plateau.

</details>

<details>
<summary>Hint for question 3: how scikit-learn decides when to stop</summary>

With `solver='sgd'`, training ends at whichever comes first:

- **The epoch limit:** `max_iter` epochs are complete.
- **The stopping rule:** after each epoch $t$, scikit-learn compares that epoch's training loss $L_t$ with $L_{\text{best}}$, the lowest training loss of any earlier epoch. The epoch counts as **no improvement** if
$$L_t + \texttt{tol} > L_{\text{best}},$$
that is, if it didn't beat the best loss so far by at least `tol` (a setting, think "tolerance"). An epoch that does improve resets the count to 0. Training stops once the count exceeds `n_iter_no_change`: with the defaults (`tol=1e-4`, `n_iter_no_change=10`), that's 11 non-improving epochs in a row. Nothing is printed when the rule fires (unless you set `verbose=True`), so a run that stopped early looks just like one that finished.

</details>

In [ ]:
# 2(b)
### YOUR CODE HERE


**2(b) Answer:**

_YOUR ANSWER HERE_


**2(c) · Code.** Train again with the stopping rule's settings changed so that it can't fire (see the hint). The rule fires only after more than `n_iter_no_change` non-improving epochs in a row, so setting `n_iter_no_change` equal to `max_iter` (3,000) means the run always reaches the epoch limit first. Training then continues for all 3,000 epochs, more than Wednesday's network needed to get past its plateaus. Store the fitted model as `xor_no_stop`, print its accuracy, and plot **both** runs' loss curves on one set of axes, with labels and the $\ln 2$ baseline.

**Points back to:** Part 1's `xor_net`.

**This cell takes a few seconds,** and it ends with a warning. Read the note below it.

<details>
<summary>Hint</summary>

the `xor_net` function from earlier takes any setting as a keyword: `xor_net(0, n_iter_no_change=...)`. Fit it as before. For the plot, `curves` is a list of two `loss_curve_`s, and `labels` a list of two strings.

</details>

In [ ]:
# 2(c)
### YOUR CODE HERE


**About that warning.** `ConvergenceWarning` means "training reached `max_iter` epochs before the stopping rule fired." We'll train many networks for a fixed number of epochs on purpose, so the next cell turns this warning off.

In [ ]:
warnings.filterwarnings('ignore', category=ConvergenceWarning)

**2(d) · Written. Read the curves.** Describe the two loss curves from 2(c) in detail, as if explaining the plot to someone who can't see it. Address:

1. Where do the two curves start, and where does each one end? Over the epochs where both exist, how do they compare, and why?
2. For `xor_no_stop`: over roughly which epochs is it on the plateau, and what happens to the loss once it leaves? Give approximate epochs and losses.
3. Suppose you had checked `xor_no_stop` at epoch 1,000. Could you have told from its loss curve so far that it would solve XOR? What does that say about deciding a run is finished because its loss has stopped changing?

You may add code cells to read exact values from the curves.

**Points back to:** [Training a Network, §6](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-6) ("Reading the run").

<details>
<summary>Hint: reading exact values</summary>

The plot squeezes the plateau into a flat line, so read numbers from the curve itself. `loss_curve_` is a list, and index $i$ holds epoch $i + 1$'s loss. To find the first epoch below a threshold, convert it to a NumPy array and use `np.argmax` on a boolean array, which returns the index of the first `True`:

```python
curve = np.array([0.9, 0.7, 0.4, 0.2])
print(curve[1])                       # epoch 2's loss: 0.7
print(np.argmax(curve < 0.5) + 1)     # first epoch below 0.5: 3
```

</details>


**2(d) Answer:**

_YOUR ANSWER HERE_


**Why doesn't this look like Wednesday's curve?** Wednesday's loss left $\ln 2$ within a few hundred steps and later sat on a second plateau near 0.5; `xor_no_stop` sat on $\ln 2$ for about 1,400 epochs and then dropped in one go. The network, the data, and the update rule are the same. The difference is the starting point: scikit-learn starts from smaller random weights, closer to the origin of parameter space. The optional item 3(c) shows why that matters.

## Part 3: Stuck runs, the loss surface, and more hidden units

On Wednesday, 10 random starts (think: "one set of random model parameters" = one "random start") resulted in trained models of wildly varying quality: most solved XOR, and some got **stuck** at a bad spot on the loss surface during gradient descent  ([Training a Network, §6](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-6)), resulting in very bad performance. 

Let's investigate losses that "get stuck" during gradient descent with scikit-learn.

**3(a) · Code.** The answer cell defines `seeds`, a list of the 20 seeds 0 through 19. Iterate through `seeds`, and for each seed train `xor_net(seed, n_iter_no_change=3000)`. Keep each run's `loss_curve_` in a list `random_seed_curves` and its accuracy on the XOR data in a list `acc2`. Then compute `n_solved2`, the number of runs with accuracy 1.0, and plot all 20 curves with `plot_loss_curves(random_seed_curves)`.

**Points back to:** [Training a Network, §6, "Different starting weights, different outcomes"](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-6).

**This cell takes about 10–30 seconds.**

<details>
<summary>Hint 1: the shape of the loop</summary>

Start with two empty lists. Inside a `for` loop over `seeds`: build and fit one network, then append its `loss_curve_` to one list and its `.score(X_xor, y_xor)` to the other. After the loop, count the 1.0s: `np.sum(np.array(acc2) == 1.0)`, or a generator expression.

</details>

<details>
<summary>Hint 2: a made-up loop over seeds</summary>

A loop that collects one result per `random_state`, with a different scikit-learn function:

```python
toy = np.arange(10)
toy_seeds = [0, 1, 2]
first_test = []
for seed in toy_seeds:
    train, test = train_test_split(toy, test_size=0.2, random_state=seed)
    first_test.append(int(test[0]))
print(first_test)
```
```
[2, 2, 4]
```

</details>

In [ ]:
# 3(a)
seeds = list(range(20))      # the 20 random starts: seeds 0, 1, ..., 19

### YOUR CODE HERE


In our results, 17 of the 20 starts solved XOR. The other 3 (seeds 9, 13, and 19) leveled off near a loss of 0.35, with only 59% to 64% accuracy: the same level Wednesday's stuck runs reached.

### Reading: the loss surface

*This is intuition, not a proof.*

- **The loss surface.** The training loss is a function of all the network's parameters. For our 9-parameter network, picture its graph as a surface over a 9-dimensional **parameter space**: each point in that space is one setting of all 9 weights and biases, and the height is the loss. Gradient descent walks downhill on this surface, one step per update.
- **Critical points** are where the gradient is zero, so gradient descent's update doesn't move. There are three kinds:
  - a **local minimum**: the loss rises in *every* direction;
  - a **saddle point**: the loss rises in some directions and falls in others;
  - the **global minimum**: the lowest loss anywhere. For XOR, a loss near 0.


- **Plateaus and stuck runs.** Near a saddle point the gradient is small, so the steps are small, and the loss curve goes almost flat for a while. These result in a **plateau** in the Loss-vs-epoch curves. Because one or more directions do lead downhill, gradient descent eventually finds a way down the loss surface and the loss drops, as seed 0 did in Part 2. Near a local minimum there's no downhill direction, so gradient descent stays stuck. Our stuck runs **behave like** runs that found a local minimum: their gradient stayed tiny for thousands of epochs.

- **Why more parameters help.** For a critical point to be a local minimum, the loss has to curve upward in every direction at once. Each added dimension is one more direction that might lead downhill instead. So in high-dimensional parameter spaces, most critical points with high loss tend to be saddle points, which gradient descent can escape, rather than local minima, which trap it. Extra hidden units also give the network many different weight settings that represent equally good solutions, which tends to connect the low-loss regions. This is an observed property of network loss surfaces (Dauphin et al., 2014, "Identifying and attacking the saddle point problem"), not a guarantee for every network.

Logistic regression's loss surface, by contrast, is a single bowl: one critical point, the global minimum ([Training a Network, §6](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-6)).

**3(b) · Code + Written. Predict, then run.** Repeat 3(a) with 3 hidden units: pass `hidden_layer_sizes=(3,)` to `xor_net` as well. Store the results in `curves3`, `acc3`, and `n_solved3`, and plot the curves. *Predict first:* will more, fewer, or the same number of runs get stuck? Then explain what you see, using the reading. How many parameters does the 3-unit network have?

**Points back to:** the reading above; [Training a Network, §6](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-6).

**This cell takes about 10–30 seconds.**

In [ ]:
# 3(b)
### YOUR CODE HERE


**3(b) Answer:**

_YOUR ANSWER HERE_


### 3(c) · Optional (not graded): where the start lands

The network, the data, the log-loss, and the update rule in this lab are the same as Wednesday's, so the loss surface is the same too. What differs is where gradient descent **starts** on it:

- Wednesday's `random_params` drew every weight and bias from a standard normal distribution, so many started with a magnitude near 1 or larger.
- scikit-learn draws each layer's weights and biases uniformly from $[-r, r]$, with $r = \sqrt{2 / (\text{fan-in} + \text{fan-out})}$ for sigmoid units ("Glorot initialization"). Here fan-in and fan-out are the layer's numbers of inputs and outputs: $r \approx 0.71$ for our hidden layer and $r \approx 0.82$ for the output layer. So scikit-learn's starts are much closer to the origin.

**Code worth reading.** The cell below starts scikit-learn from Wednesday's exact starting weights, multiplied by a **scale** of 2, 1, 0.3, or 0.1. Every start points in the same direction in parameter space; only its distance from the origin changes. Two helpers do the setup: `wednesday_start(scale)` returns Wednesday's seed-1 starting weights times `scale`, and `fit_from_start(model, start, X, y)` fits `model` starting from those weights instead of random ones. With scale 1, the run reproduces Wednesday's loss curve exactly.

*Predict first:* which scale will leave the loss $= \ln 2 \approx 0.7$ plateau first? Which will reach a loss of 0.1 first?

**This cell takes about 5 seconds.**

In [ ]:
scales = [2, 1, 0.3, 0.1]
runs_by_scale = []
for scale in scales:
    start = wednesday_start(scale)                               # Wednesday's starting weights, times scale
    network = xor_net(0, max_iter=5000, n_iter_no_change=5000)   # n_iter_no_change = max_iter: the stopping rule can't fire; the seed doesn't matter here
    runs_by_scale.append(fit_from_start(network, start, X_xor, y_xor))

for scale, run in zip(scales, runs_by_scale):
    curve = np.array(run.loss_curve_)
    print(f'scale {scale}: first below 0.6 at epoch {np.argmax(curve < 0.6) + 1}, '
          f'first below 0.1 at epoch {np.argmax(curve < 0.1) + 1}, accuracy {run.score(X_xor, y_xor)}')

plot_loss_curves([run.loss_curve_ for run in runs_by_scale], labels=[f'scale {s}' for s in scales],
                 baseline=np.log(2), title="Wednesday's starting weights, scaled");

In our results, all four solved XOR. Scale 2 fell below 0.6 at epoch 182 and scale 0.1 not until epoch 4,218. Scales 2 and 1 then sat on a second plateau near 0.5 for roughly 1,000 and 1,800 epochs; scales 0.3 and 0.1 dropped in one go.

#### Reading (Optional continued): why the starting point matters

*This is intuition, not a proof.*

- **Near the origin, the network is almost linear.** With small weights, every hidden unit's input $z$ is near 0, where the sigmoid is nearly a straight line: $\sigma(z) \approx \tfrac12 + \tfrac{z}{4}$. Then each hidden activation is approximately a linear function of $x$, and the whole network behaves like logistic regression. No line separates XOR, so the best logistic regression can do is about $p = 0.5$ everywhere: a loss of $\ln 2$. That's why small starts sit on the loss $= \ln 2 \approx 0.7$ plateau. It's a saddle region: flat in most directions, with a way down that uses the sigmoid's curve.
- **Smaller starts take longer to escape.** The sigmoid's curve only matters once the weights are large enough to push $z$ away from 0. Near the origin, the slope toward that region is small because the weights are small, so the closer a run starts to the origin, the longer it spends on the plateau.
- **Small starts tend to drop in one go.** During the slow escape, gradient descent grows the weights mainly along the direction that leads downhill fastest, while the rest of the random start stays small. By the time the weights matter, both hidden units have been shaped by the same useful signal. A large start has already committed its hidden units to arbitrary features, and gradient descent has to work from those, which can lead it past other saddles (like the plateau near 0.5 at scales 1 and 2) or into a trap. This is the standard explanation from studies of saddle points in networks, and it's a tendency, not a guarantee: seeds 9, 13, and 19 in 3(a) started small and still got stuck.
- **What the trap's loss means.** The loss averages over the data, and each XOR cluster holds a quarter of the points. The stuck runs in 3(a) classify two of the four clusters confidently (loss near 0 on each) and predict $p = 0.5$ on the other two (loss $\ln 2$ on each), so their loss is about $(0 + 0 + \ln 2 + \ln 2)/4 = \ln 2 / 2 \approx 0.35$.

**Question (Optional).** All four runs used the same network, data, loss surface, and update rule. What differed between them? Is starting closer to the origin faster or slower?

<details>
<summary>Answer</summary>

Only the starting point differed: the same direction in parameter space, at different distances from the origin. The smaller the start, the longer the run stayed on the loss $= \ln 2 \approx 0.7$ plateau. But neither small nor large is simply faster. Scales 1 and 2 left $\ln 2$ quickly and then stalled on a second plateau; scale 2 still reached a loss of 0.1 first (epoch 1,499). Scale 0.3 dropped in one go and reached 0.1 at epoch 2,014, before scale 1 did (epoch 2,520). Scale 0.1 started so close to the origin that it took until epoch 4,461. Where a random start lands on the loss surface, relative to its plateaus and traps, shapes the whole run. That's why the same network trains so differently from seed to seed.

</details>

## Part 4: Minibatches

Wednesday's minibatch gradient descent took one step per batch of points, and an **epoch** was one pass through all the data ([Training a Network, §7](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-7)). In `MLPClassifier`, `max_iter` counts **epochs**, and two attributes count the work done: `n_iter_` is the number of epochs completed, and `t_` is the total number of training points processed, summed over all epochs. Each epoch processes all $n$ points once, whatever the batch size.

Every run so far has been **full batch**, meaning each gradient step computed the gradient with every point in the training set (with `batch_size=100`, each gradient step used all $n = 100$ points).


**A full-batch run to compare against.** The cell below trains the full-batch network you've been using, from seed 3. In 4(a) and 4(b) you'll train the same network from the same start with minibatches instead, and compare the two runs. (We use seed 3 because its minibatch run shows the comparison clearly.) Two settings matter for that comparison:

- `max_iter=300` sets the budget: each run gets 300 epochs, that is, 300 passes through the data. That's a small budget: in Part 2, the full-batch run from seed 0 spent about 1,400 epochs on the loss $= \ln 2 \approx 0.7$ plateau.
- `n_iter_no_change=300` keeps the stopping rule from firing. The rule is still on, but it would need more than 300 non-improving epochs in a row to trigger, and training ends after epoch 300. So each run uses its whole budget of 300 epochs, even if its loss sits on a plateau.

In [ ]:
xor_full_batch = xor_net(seed = 3, max_iter=300, n_iter_no_change=300).fit(X_xor, y_xor)
print('full batch: epochs', xor_full_batch.n_iter_, ' accuracy', xor_full_batch.score(X_xor, y_xor))

**Counting epochs and gradient steps.** The cell below trains the same network from the same start with minibatches of 10 points (`batch_size=10`), and stores it as `xor_minibatch`. Then, for both models, it prints the counts scikit-learn keeps and works out the number of gradient steps from them.

**Points back to:** [Training a Network, §7](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-7).

In [ ]:
xor_minibatch = xor_net(3, max_iter=300, n_iter_no_change=300, batch_size=10).fit(X_xor, y_xor)

n = len(y_xor)                                       # 100 training points
for name, model in [('full batch', xor_full_batch), ('batch size 10', xor_minibatch)]:
    steps_per_epoch = n // model.batch_size          # one gradient step per batch
    total_steps = model.t_ // model.batch_size       # t_: training points processed, over all epochs
    print(f'{name} (batch_size={model.batch_size}):')
    print(f'  epochs, n_iter_:               {model.n_iter_}')
    print(f'  training points processed, t_: {model.t_:,} = {model.n_iter_} epochs x {n} points')
    print(f'  gradient steps per epoch:      {steps_per_epoch} = {n} points / {model.batch_size} points per step')
    print(f'  total gradient steps:          {total_steps:,} = {model.n_iter_} epochs x {steps_per_epoch} per epoch')

**Reading the output.** An epoch is one pass through all 100 training points, so after 300 epochs both models have processed $300 \times 100 = 30{,}000$ training points: that's `t_`. What differs is how those points are grouped into gradient steps:

- **Full batch:** each step uses all 100 points, so there's 1 step per epoch, and 300 steps in all. For full batch, epochs and steps are the same count.
- **Batch size 10:** each step uses only 10 points, so there are $100 / 10 = 10$ steps per epoch, and $300 \times 10 = 3{,}000$ steps in all.

So `n_iter_` counts **epochs** (passes through the data), and the number of gradient steps counts **updates to the weights**. In general, $\text{total gradient steps} = \text{epochs} \times \dfrac{n}{\text{batch size}}$.

**4(a) · Written.** Use the formula above; you don't need to train anything.

1. Suppose you trained the same network with `batch_size=25` for 300 epochs. How many training points would it process (`t_`), how many gradient steps would it take per epoch, and how many in total?
2. Suppose you wanted the full-batch model to take as many gradient steps as `xor_minibatch` did (3,000). How many epochs would it need, and how many training points would it process? Compare that with `xor_minibatch`'s 30,000.

**4(a) Answer:**

_YOUR ANSWER HERE_


**Two views of the same two runs.** The cell below plots both models' loss curves twice: first against epochs, then against gradient steps. Run it; you don't need to change anything.

One detail: for minibatches, `loss_curve_` records the average of the batch losses *during* each epoch, not the loss on all the data at the end of it. That's close enough for comparing the two models.

**Points back to:** [Training a Network, §7](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-7).

In [ ]:
full_batch_losses = xor_full_batch.loss_curve_      # one loss per epoch: 300 entries
minibatch_losses = xor_minibatch.loss_curve_        # one loss per epoch: 300 entries
labels = ['full batch', 'batch size 10']

# Plot 1: the x-axis counts epochs (passes through the data).
plot_loss_curves([full_batch_losses, minibatch_losses], labels=labels, baseline=np.log(2), title='By epoch');

# Plot 2: the x-axis counts gradient steps. One epoch is 1 step for full batch and 10 steps for batch size 10.
plot_loss_curves([full_batch_losses, minibatch_losses], labels=labels, x='step', steps_per_epoch=[1, 10],
                 baseline=np.log(2), title='By gradient step');

**4(b) · Written.**

1. In each plot, which model has the lower loss at the point where the full-batch curve ends (epoch 300 in the first plot, step 300 in the second)?
2. Both models processed the same 30,000 training points (4(a)). Which plot compares the two methods at equal computational cost?, By that comparison, which method was more effective?

**4(b) Answer:**

_YOUR ANSWER HERE_


**Code worth reading.** This cell trains 20 models: for each seed from 0 to 9, one full-batch and one minibatch, 300 epochs each. **It takes about 10–30 seconds.**

In [ ]:
print('seed   full batch accuracies   batch size 10 accuracies')
for seed in range(10):
    f = xor_net(seed, max_iter=300, n_iter_no_change=300).fit(X_xor, y_xor)
    m = xor_net(seed, max_iter=300, n_iter_no_change=300, batch_size=10).fit(X_xor, y_xor)
    print(f'{seed:4d}   {f.score(X_xor, y_xor):10.2f}   {m.score(X_xor, y_xor):13.2f}')

**4(c) · Written.** For a fixed budget of 300 epochs, which method wins, and by how much? Does minibatch training always solve XOR?

**4(c) Answer:**

_YOUR ANSWER HERE_


## Part 5: Digits, a 10-class problem

Now a real classification problem: scikit-learn's handwritten digits. Each image is $8 \times 8$ pixels, stored as 64 features with values from 0 (white) to 16 (black), and the label is the digit, 0 to 9. Run the cell below.

In [ ]:
digits = load_digits()
Xd, yd = digits.data, digits.target
print(Xd.shape, 'images per class:', np.bincount(yd))

# Each row of Xd is one 8 x 8 image, flattened into 64 pixel values. To draw an image,
# reshape its row back into an 8 x 8 array and pass that to imshow.
fig, axes = plt.subplots(1, 10, figsize=(13, 1.8))      # one row of 10 axes, one per image
for ax, pixels, label in zip(axes, Xd[:10], yd[:10]):
    ax.imshow(pixels.reshape(8, 8), cmap='gray_r')     # gray_r: high values (dark ink) drawn dark
    ax.set_title(str(label))
    ax.axis('off')                                     # no ticks or frame around an image
fig.suptitle('The first ten images')
plt.show()

Xd_train, Xd_test, yd_train, yd_test = train_test_split(Xd, yd, test_size=0.25, stratify=yd, random_state=0)
print('training images:', len(yd_train), ' test images:', len(yd_test))

**What's different about a 10-class network.**

- **The output layer has one unit per class:** 10 here. `predict_proba` returns 10 probabilities per image, one per digit, and they sum to 1. (How the output layer produces probabilities that sum to 1 is Monday's topic, the *softmax*. For now, `predict` picks the most probable digit.)
- **The defaults change three things from XOR.** We'll use `MLPClassifier`'s default settings here:
  - 100 hidden units;
  - the ReLU activation, $\max(0, z)$, from [Neurons to Networks, §1](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_02_neurons_to_networks.ipynb#section-1);
  - the `adam` solver: a variant of gradient descent that adapts each parameter's step size as it goes. The slopes still come from backpropagation.
- **The default stopping rule is fine here.** Part 2's trap was XOR's long plateau at the start, and this problem doesn't have one.

`make_pipeline(StandardScaler(), MLPClassifier(...))` standardizes each pixel (using the training images' means and standard deviations) before the network sees it, as in Chapter 4.

**5(a) · Code + Written.** Create the pipeline `digits_net = make_pipeline(StandardScaler(), MLPClassifier(random_state=0))`, fit it on the training images, and print its test accuracy. Then print the shapes of its network's `coefs_`, and compute `digits_params`, its number of weights and biases. How many output units does it have, and why that many?

**Points back to:** Part 1(a); [Neurons to Networks, "Following the data through the network"](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_02_neurons_to_networks.ipynb#forward-pass) (a layer maps one dimension per input to one dimension per unit).

**This cell takes a few seconds.**

<details>
<summary>Hint: reaching the network inside a pipeline</summary>

A pipeline can be indexed like a list: `pipe[-1]` is its last step. For example, on made-up data:

```python
pipe = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_toy, y_toy)   # X_toy: 20 points, 4 features
print(pipe[-1])
print(pipe[-1].coef_.shape)
```
```
LogisticRegression()
(1, 4)
```

</details>

In [ ]:
# 5(a)
### YOUR CODE HERE


**5(a) Answer:**

_YOUR ANSWER HERE_


**5(b) · Code + Written.** Print `digits_net`'s `predict_proba` row for the first test image, `Xd_test[:1]`, rounded to 3 places. Which digit does it favor, how confident is it, and do the 10 probabilities sum to 1? Compare with that image's label, `yd_test[0]`.

<details>
<summary>Hint</summary>

`predict_proba` expects a 2-D array, one row per image. `Xd_test[:1]` (a slice) keeps the 2-D shape, but `Xd_test[0]` doesn't and raises an error. On a made-up 2-class pipeline:

```python
probs = pipe.predict_proba(X_toy[:1])
print(probs.round(3), probs.shape)
print(probs.sum(), probs.argmax())
```
```
[[0.474 0.526]] (1, 2)
1.0 1
```

</details>

In [ ]:
# 5(b)
### YOUR CODE HERE


**5(b) Answer:**

_YOUR ANSWER HERE_


**Where does the network go wrong?** The cell below is written for you; just run it. It predicts every test image with `digits_net`, then shows two things:

- **The confusion matrix.** Row $i$, column $j$ counts the test images whose true digit is $i$ and that the network predicted as $j$. The diagonal counts the correct predictions; every number off the diagonal is a mistake.
- **Every misclassified test image,** titled "true → predicted".

In [ ]:
# Uses your digits_net from 5(a). You don't need to change anything here.
pred = digits_net.predict(Xd_test)
plot_confusion(yd_test, pred)
wrong = np.flatnonzero(pred != yd_test)
print(len(wrong), 'of', len(yd_test), 'test images misclassified')
show_digits(Xd_test[wrong], yd_test[wrong], predicted=pred[wrong], title='Misclassified: true → predicted')   # Part 5's imshow loop, in a grid

**5(c) · Written.** Look at the misclassified images. Which of the mistakes look forgivable, ones a person might make too? Which look like mistakes a person wouldn't make?

**5(c) Answer:**

_YOUR ANSWER HERE_


**For comparison: logistic regression** (multinomial, from Chapter 4), on the same split. It has one weight per pixel plus an intercept for each of the 10 classes: $65 \times 10 = 650$ parameters. Part 6 comes back to this number.

In [ ]:
digits_logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xd_train, yd_train)
print('8x8 digits, logistic regression test accuracy:', round(digits_logreg.score(Xd_test, yd_test), 3))

**5(d) · Written.** Logistic regression just scored almost the same accuracy of our neural network. Why do you think this is possible (there are no wrong answers... what do you think?). Hint: Were these images representative of typical hand-written digits you'd encounter, say, scanning a check?



**5(d) Answer:**

_YOUR ANSWER HERE_


## Part 6: Harder digits (MNIST), and how wide the hidden layer must be

The 8 × 8 digits are an easy problem: logistic regression already gets about 97%. **MNIST** is the standard harder version, and it's the dataset ISL §10.2 uses: $28 \times 28$ pixel images, 784 features with values 0 to 255, 60,000 training images and 10,000 test images. 

**Code worth reading.** The data is in `mnist.npz` in this folder, a NumPy archive of four arrays. To keep training times short, we train on a random 10,000 of the 60,000 training images, and test on all 10,000 test images. Each image is flattened into one row of 784 features, and divided by 255 so the pixels range from 0 to 1.

In [ ]:
data = np.load('mnist.npz')                      # relative path: mnist.npz must sit next to this notebook
print({name: data[name].shape for name in data.files})

rng = np.random.default_rng(0)
keep = rng.choice(60000, size=10000, replace=False)           # a random 10,000 of the training images
X_train = data['x_train'][keep].reshape(-1, 784) / 255         # each 28 x 28 image becomes one row of 784
y_train = data['y_train'][keep]
X_test = data['x_test'].reshape(-1, 784) / 255
y_test = data['y_test']
print('training:', X_train.shape, ' test:', X_test.shape)

fig, axes = plt.subplots(1, 10, figsize=(13, 1.8))
for ax, pixels, label in zip(axes, X_train[:10], y_train[:10]):
    ax.imshow(pixels.reshape(28, 28), cmap='gray_r')   # same as Part 5, but each row is 28 x 28 = 784 pixels
    ax.set_title(str(label))
    ax.axis('off')
fig.suptitle('Ten MNIST training images')
plt.show()

**Logistic regression on MNIST,** for reference: $(784 + 1) \times 10 = 7850$ parameters. **This cell takes about 5–20 seconds.**

In [ ]:
mnist_logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
logreg_train = mnist_logreg.score(X_train, y_train)
logreg_test = mnist_logreg.score(X_test, y_test)
print(f'logistic regression: training accuracy {logreg_train:.3f}, test accuracy {logreg_test:.3f}')

**Width.** A network with $h$ hidden units maps each 784-pixel image to $h$ numbers, a point in an $h$-dimensional h-space, and its output layer then draws the class boundaries in that space ([Neurons to Networks, "Following the data through the network"](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_02_neurons_to_networks.ipynb#forward-pass)). How large does $h$ need to be?

**6(a) · Code. The width sweep.** For each width in `widths`, fit `MLPClassifier(hidden_layer_sizes=(h,), random_state=0, max_iter=100)` on `X_train, y_train` (no pipeline needed; the pixels are already scaled). Keep each fitted network in the dict `mnist_nets` (keyed by width), and its training and test accuracy in the lists `train_acc` and `test_acc`. The plot at the end of the cell runs once your code has filled those in.

**Points back to:** Part 3's loop over seeds (the same pattern, now over widths).

**This cell takes about 20–60 seconds.**

<details>
<summary>Hint: a skeleton</summary>

```python
for h in widths:
    model = MLPClassifier(hidden_layer_sizes=..., random_state=0, max_iter=100).fit(..., ...)
    mnist_nets[h] = model
    train_acc.append(model.score(..., ...))
    test_acc.append(model.score(..., ...))
```

</details>

In [ ]:
# 6(a)
widths = [1, 2, 5, 20, 100]
mnist_nets, train_acc, test_acc = {}, [], []
### YOUR CODE HERE
for h, tr, te in zip(widths, train_acc, test_acc):
    print(f'{h:4d} hidden units: training {tr:.3f}, test {te:.3f}')
plot_accuracy_vs_width(widths, train_acc, test_acc, reference=logreg_test)

**Inside two networks' hidden layers.** Each panel plots the 10,000 test images, colored by true digit, at two hidden activations. It uses your `mnist_nets` from 6(a).

- **Left: the 2-unit network.** Its hidden layer has only 2 units, so $(h_1, h_2)$ is its whole h-space: this is everything its output layer sees.
- **Right: the 20-unit network.** Its h-space has 20 dimensions, which we can't draw, so this panel shows just 2 of them, $h_1$ and $h_2$. Its output layer sees all 20.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_h_space(mnist_nets[2], X_test, y_test, ax=axes[0], title='2-unit network: its whole h-space')
plot_h_space(mnist_nets[20], X_test, y_test, units=(0, 1), ax=axes[1], title='20-unit network: 2 of its 20 hidden units')
plt.tight_layout()
plt.show()

**6(b) · Written.** Use your table, the accuracy plot, and the h-space plot.

1. Describe how test accuracy changes as the hidden layer gets wider. Where does logistic regression's line cross the network's?
2. The 2-unit network's **training** accuracy is low too, far below logistic regression's. Is its problem overfitting, or something else? What does the low training accuracy tell you?
3. A network is supposed to be the more flexible model. How can logistic regression on the 784 pixels beat a network with a 2-unit hidden layer?
4. The right panel (h-space plots) looks even *more* mixed up than the left, yet the 20-unit network is far more accurate. How can both be true? If you could see all 20 dimensions of its h-space, what would you expect the 10 digits to look like?

**Points back to:** [Neurons to Networks, "Following the data through the network"](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_02_neurons_to_networks.ipynb#forward-pass) and the reminder above.

<details>
<summary>Hint: what to look at</summary>

Count how many numbers per image reach each model's final decision: logistic regression's, and the 2-unit network's output layer. Where do the 10 digits sit in the h-space plot? For question 2, remember what overfitting looks like: good training accuracy, worse test accuracy. For question 4, the output layer is multinomial logistic regression on $h$, so it separates the digits with flat boundaries. What would its training accuracy be if the digits overlapped in all 20 dimensions the way they do in the right panel?

</details>

**6(b) Answer:**

_YOUR ANSWER HERE_


**Check your answer to question 4.** The output layer can only draw flat boundaries, so you can measure how well flat boundaries separate the digits in any set of hidden activations: fit logistic regression on them and look at its accuracy. The cell below computes each network's hidden activations directly, $h = \max(0,\ x W + b)$ (the first layer of the forward pass, with scikit-learn's default ReLU activation), then fits logistic regression on three versions of $h$. Run it after you've answered 6(b).

In [ ]:
def hidden_layer(net, X):
    "Each row of X's hidden activations: the first layer of the forward pass, with ReLU."
    return np.maximum(0, X @ net.coefs_[0] + net.intercepts_[0])


H2_train, H2_test = hidden_layer(mnist_nets[2], X_train), hidden_layer(mnist_nets[2], X_test)        # shape (n, 2)
H20_train, H20_test = hidden_layer(mnist_nets[20], X_train), hidden_layer(mnist_nets[20], X_test)    # shape (n, 20)

versions = [('2-unit network, both units (left panel)', H2_train, H2_test),
            ('20-unit network, units 1 and 2 (right panel)', H20_train[:, :2], H20_test[:, :2]),
            ('20-unit network, all 20 units', H20_train, H20_test)]
for name, H_train, H_test in versions:
    flat = LogisticRegression(max_iter=2000).fit(H_train, y_train)
    print(f'{name:46s} training {flat.score(H_train, y_train):.3f}   test {flat.score(H_test, y_test):.3f}')

In our results, logistic regression on the right panel's 2 coordinates scored only 0.333 on the training images, below the 2-unit network's 0.678, but on all 20 coordinates it scored 0.994. The separation is there, but only when you use all 20 dimensions.

**6(c) · Written. Pick and defend.** A teammate says: "Neural networks are much better than logistic regression at recognizing handwritten digits." Agree or push back, using three comparisons:

| Data | Logistic regression | Network | Source |
|---|---|---|---|
| 8 × 8 digits, 1,347 training images | test accuracy from Part 5 | test accuracy from 5(a) | your results above |
| MNIST, 10,000 training images | test accuracy above | 100-unit network from 6(a) | your results above |
| MNIST, all 60,000 training images | 7.2% test error | 1.8% test error (with a regularization method from Monday) | ISL Table 10.1 |

Context for the 8 × 8 digits, from the dataset's description (`print(digits.DESCR)`): each 8 × 8 pixel is a count of the dark pixels in a 4 × 4 block of an original 32 × 32 image, which "reduces dimensionality and gives invariance to small distortions." MNIST's 784 features are the raw pixels.

Say what the network costs, too.

**Points back to:** Parts 2, 5, and 6.

<details>
<summary>Hint: what to look at</summary>

Compare error rates (1 − accuracy), not just accuracies: going from 3% error to 2% removes a third of the mistakes. Then ask what the 8 × 8 preprocessing already does that a hidden layer would otherwise have to learn.

</details>

**6(c) Answer:**

_YOUR ANSWER HERE_


## Wrap-up

- **Part 2:** `MLPClassifier`'s default stopping rule ends training when the loss stops changing, which can happen on a plateau. Look at the loss curve and the accuracy, not just "it finished" ([Training a Network, §6](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-6)).
- **Part 3:** a network's loss surface isn't a bowl. Some starts get stuck; more hidden units give more directions downhill.
- **Part 4:** at equal computation (epochs), minibatches take many more steps than "full" batches and usually get further downhill on the loss surface ([§7](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_03_training_networks.ipynb#section-7)).
- **Parts 5–6:** a hidden layer too narrow for the problem is a bottleneck that underfits. A network's advantage over logistic regression depends on the problem: large on raw pixels, small on well-engineered features.


## Optional (not graded)

These go further, for anyone with time left. Each has its code written for you and a folded-up answer.

### O1. Why does shuffling the folds matter?

Cross-validation estimates test accuracy by splitting the data into $K$ folds and testing on each in turn (Chapter 5). There are two common ways to make the folds:

- **Unshuffled** (`cv=5` with a classifier): `StratifiedKFold(5)` without shuffling. Each fold is a contiguous block of rows, in the order they appear in the file, with the classes balanced across folds.
- **Shuffled**: `StratifiedKFold(5, shuffle=True, random_state=0)`. The rows are scattered at random across the folds (still class-balanced), so neighbors in the file usually land in different folds.

**How the 8 × 8 digits were collected** (from `digits.DESCR`): the images were extracted from preprinted forms that people filled in by hand. This file is the test set of a larger UCI dataset, and its 1,797 images were written by **13 people**. The description doesn't say how the rows are ordered.

**This cell takes about 10–20 seconds.**

In [ ]:
models = {'logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
          'network (100 units)': make_pipeline(StandardScaler(), MLPClassifier(random_state=0, max_iter=500))}
shuffled = StratifiedKFold(5, shuffle=True, random_state=0)
for name, model in models.items():
    plain = cross_val_score(model, Xd, yd, cv=5)
    mixed = cross_val_score(model, Xd, yd, cv=shuffled)
    print(f'{name:20s} unshuffled {plain.round(3)} mean {plain.mean():.3f}')
    print(f'{"":20s} shuffled   {mixed.round(3)} mean {mixed.mean():.3f}')

**Questions.** Why might the unshuffled folds give lower and more variable accuracy than the shuffled folds? What would have to be true about the order of the rows in the file for that to happen? If you wanted to predict accuracy on digits written by a **new** person, which estimate would you trust, and why?

<details>
<summary>Answer</summary>

If each writer's images sit together in the file (for example, if the rows are grouped by the form they came from), then each unshuffled fold is mostly made of a few writers whose handwriting the model saw little of during training, which is harder. Shuffled folds put every writer's images in every fold, so the model has always seen some of each test writer's handwriting, which makes the task easier and the estimate optimistic. For a **new** writer, the unshuffled estimate is the more honest one. (The row order isn't documented, so this is an inference from the scores. Grouped cross-validation, scikit-learn's `GroupKFold`, handles this directly when the groups are known.)

</details>

### O2. The learning rate

The 3-unit XOR network from 3(b), seed 0, with three learning rates $\eta$. **This cell takes a few seconds.**

In [ ]:
runs = [xor_net(0, hidden_layer_sizes=(3,), n_iter_no_change=3000, learning_rate_init=lr).fit(X_xor, y_xor)
        for lr in (0.01, 1.0, 20.0)]
for lr, run in zip((0.01, 1.0, 20.0), runs):
    print(f'eta = {lr:5}: final loss {run.loss_curve_[-1]:.3f}, accuracy {run.score(X_xor, y_xor):.2f}')
plot_loss_curves([run.loss_curve_ for run in runs], labels=['eta = 0.01', 'eta = 1', 'eta = 20'],
                 baseline=np.log(2), title='Three learning rates');

**Question.** What goes wrong with the smallest and the largest learning rates?

<details>
<summary>Answer</summary>

With $\eta = 0.01$ the steps are tiny: in our results the loss never left the plateau at 0.693 in 3,000 epochs. With $\eta = 20$ the steps overshoot: the loss jumped well above where it started (as high as 4.67) and ended at 1.285. $\eta = 1$ solved it. As in Chapter 4's gradient descent demos, the learning rate has to be small enough not to overshoot, and large enough to make progress.

</details>

### O3. Does standardizing the 8 × 8 pixels matter?

The default network from 5(a), with and without `StandardScaler`, over five seeds. **This cell takes about 10–20 seconds.**

In [ ]:
for seed in range(5):
    raw = MLPClassifier(random_state=seed).fit(Xd_train, yd_train).score(Xd_test, yd_test)
    scaled = make_pipeline(StandardScaler(), MLPClassifier(random_state=seed)).fit(Xd_train, yd_train).score(Xd_test, yd_test)
    print(f'seed {seed}: raw pixels {raw:.3f}   standardized {scaled:.3f}')

**Question.** Standardizing doesn't change what the images show. Why might it still help a network trained by gradient descent?

<details>
<summary>Answer</summary>

In our results, standardized pixels gave 0.976–0.980 test accuracy and raw pixels 0.960–0.976. Gradient descent uses one learning rate for every weight. When features have very different scales (raw pixels range from 0 to 16, and some are almost always 0), the loss is much steeper along some weights than others, so one step size is too big for some directions and too small for others. Standardizing puts the features on a common scale, which makes the loss surface better conditioned for gradient descent. Adam's per-parameter step sizes reduce this problem, but don't remove it.

</details>

### O4. XOR's h-space

In Neurons to Networks, the hand-set XOR network's hidden layer moved the points so that one line in h-space could separate the classes ([Neurons to Networks, "Following the data through the network"](../../Course_Documents/Slides/Ch_6_and_10/ch_6_and_10_02_neurons_to_networks.ipynb#forward-pass)). Here is the same picture for a network that gradient descent trained: `xor_no_stop` from 2(c), with each XOR point at its two hidden activations and the output unit's line $p = 0.5$.

In [ ]:
plot_h_space(xor_no_stop, X_xor, y_xor, title="XOR in xor_no_stop's h-space")

**Question.** In 6(b), 2 hidden units were far too few. Why are 2 enough here?

<details>
<summary>Answer</summary>

XOR has only 2 features and 2 classes, and 2 hidden units are enough to move its four clusters so that one line separates them: the output layer needs only one boundary, and the classes land on opposite sides of it. MNIST has 784 features and 10 classes; squeezing those images into 2 numbers loses most of what tells the digits apart. How wide a hidden layer must be depends on the problem.

</details>